In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../../")
sys.path.append('../../src')
sys.path.append('/home/julian/BEPiezo-Learn/src')

%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import random
import matplotlib.pyplot as plt

from m3util.ml.rand import set_seeds
from m3util.viz.style import set_style
from m3util.viz.printing import printer
from belearn.viz.viz import Viz
from belearn.dataset.dataset import BE_Dataset
from belearn.functions.sho import SHO_nn

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

from m3util.viz.layout import inset_connector, add_box
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front

from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

printing = printer(basepath = './Figures/')


set_style("printing")
set_seeds(seed=42)

%matplotlib inline

In [ ]:
def SHO_fit_func_nn(params,
                    wvec_freq,
                    device='cpu'):
    """_summary_

    Returns:
        _type_: _description_
    """

    Amp = params[:, 0].type(torch.complex128)
    w_0 = params[:, 1].type(torch.complex128)
    Q = params[:, 2].type(torch.complex128)
    phi = params[:, 3].type(torch.complex128)
    wvec_freq = torch.tensor(wvec_freq)

    Amp = torch.unsqueeze(Amp, 1)
    w_0 = torch.unsqueeze(w_0, 1)
    phi = torch.unsqueeze(phi, 1)
    Q = torch.unsqueeze(Q, 1)

    wvec_freq = wvec_freq.to(device)

    numer = Amp * torch.exp((1.j) * phi) * torch.square(w_0)
    den_1 = torch.square(wvec_freq)
    den_2 = (1.j) * wvec_freq.to(device) * w_0 / Q
    den_3 = torch.square(w_0)

    den = den_1 - den_2 - den_3

    func = numer / den

    return func

# maybe add a divide_by_2 flag for this because I don't want it for figure 3 but maybe need it for this figure? 

In [ ]:
# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

In [ ]:
#dataset.SHO_Scaler()

In [ ]:
#h5_loop_fit, h5_loop_group = dataset.LSQF_Loop_Fit()

In [ ]:
%load_ext autoreload
%autoreload 2

from belearn.dataset.dataset import BE_Dataset
from belearn.viz.viz import Viz
from m3util.viz.printing import printer
printing = printer(basepath = './Figures/')

In [ ]:
# instantiate the visualization object
#image_scalebar = [2000, 500, "nm", "br"]

In [ ]:

# BE_viz = Viz(dataset, printing, verbose=True, 
#              SHO_ranges = [(0,1.5e-4), (1.31e6, 1.33e6), (-300, 300), (-np.pi, np.pi)], 
#              image_scalebar = image_scalebar)  

In [ ]:
# instantiates the visualization object
BE_viz = Viz(dataset, printing, verbose=True)

In [ ]:

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn

device = 'cuda:0'

datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
                            # BE_viz.loop_fitting_function_torch, # function 
                            hysteresis_nn,  # function
                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/julian/Alibek_BEPFM/Rapid-Fitting-BEPFM-NN/notebooks/6_Figures_fig4_test.ipynb")

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 100}


train =  False

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.005630633379850123.pth"
    )

In [ ]:

#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn

device = 'cuda:0'

datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
                            # BE_viz.loop_fitting_function_torch, # function 
                            hysteresis_nn,  # function
                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/julian/Alibek_BEPFM/Rapid-Fitting-BEPFM-NN/notebooks/6_Figures_fig4_test.ipynb")

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 10} #100


train =  False

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 15, # 500
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       #"./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.005630633379850123.pth"
        "./SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_14_train_loss_0.016973057058122423.pth")

In [ ]:

#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn

device = 'cuda:0'

datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
                            # BE_viz.loop_fitting_function_torch, # function 
                            hysteresis_nn,  # function
                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/julian/Alibek_BEPFM/Rapid-Fitting-BEPFM-NN/notebooks/6_Figures_fig4_test.ipynb")

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 10} #100


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 15, # 500
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       #"./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.005630633379850123.pth"
        "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_14_train_loss_0.013669469517966111.pth"    )

In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn

device = 'cuda:0'

datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
                            # BE_viz.loop_fitting_function_torch, # function 
                            hysteresis_nn,  # function
                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/julian/Alibek_BEPFM/Rapid-Fitting-BEPFM-NN/notebooks/6_Figures_fig4_test.ipynb")

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 100}


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.00602861393450035.pth"
    )

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

pred_recon, pred_params_scaled, pred_params = model.predict(
    data,
    1024,
    translate_params=False,
    is_SHO=False
)
# ran after  only 15 epochs of training on Jan 31 
fig = BE_viz.hysteresis_maps(pred_params, cycle=0, filename="Figure_XX_NN_Hysteresis_Maps")

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)

fig = BE_viz.violin_plot_comparison_hysteresis(model,
                                         torch.atleast_3d(torch.tensor(data.reshape(-1, 96))),
                                         filename="Figure_XX_Violin_test") 

In [ ]:
n = 1

# JGODDY: cannot redefine data because it must be from dataset.get_hysteresis so just put this data tuple directly into 
# BE_viz.hysteresis_comparison without predefining it 
 
#data = ("LSQF", "NN") 

fig = BE_viz.hysteresis_comparison(
    ("LSQF", "NN"),
    nn_model=model,
    filename="Figure_XX_LSQF_NN_bmw_comparison_loaded_weights_15_epochs",
)

In [ ]:
axes=fig.axes

In [ ]:
import pandas as pd
import seaborn as sns
import itertools
from torch import nn


from m3util.viz.layout import imagemap, FigDimConverter, subfigures
from m3util.viz.text import number_to_letters, set_sci_notation_label, labelfigs, bring_text_to_front

from matplotlib.ticker import ScalarFormatter
#from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import EngFormatter
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec





def y_formatter(y, pos):
    return f"{y * 1e3:.1f}"  # Multiply by 1e3 to show scaled values

def copy_axes_properties(row,col,source_ax, target_ax, secondary_ax, ax_lims):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    #target_ax.set_xlabel(source_ax.get_xlabel())
    # uncomment out below 
    target_ax.set_xlim([-20,20])

    if row == 2: # if i in [4,5]:
        target_ax.set_xlabel('Frequency (MHz)',fontsize=20)
    else:
        target_ax.set_xlabel("")
        #target_ax.set_xticks([])
        target_ax.xaxis.set_ticklabels([])

   #target_ax.set_ylim(ax_lims.get_ylim() if max(ax_lims.get_ylim()) > max(source_ax.get_ylim()) else source_ax.get_ylim())

    if col == 0:    
        #target_ax.set_ylim(source_ax.get_ylim())
        target_ax.set_ylabel(source_ax.get_ylabel(),fontsize=20)
    else:
        #target_ax.set_ylim(ax_lims.get_ylim() if max(ax_lims.get_ylim()) > max(source_ax.get_ylim()) else source_ax.get_ylim())
        target_ax.set_ylabel("")
        target_ax.yaxis.set_ticklabels([])

        
    # if row == 0:
    #     target_ax.set_ylim([-0.5e-3,8.0e-3])
    # elif row == 1: 
    #     target_ax.set_ylim([-0.1e-2,2.1e-2])
    # elif row == 2: 
    #     target_ax.set_ylim([-0.1e-2,2.1e-2])    
    
    #target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        
    # Handle twin axes if present
    #if secondary_ax:
    ax_twin = target_ax.twinx()
    ax_twin.set_ylim(secondary_ax.get_ylim())
    #ax_twin.set_yticks([-np.pi,0,np.pi],labels =["$-\pi$","0","$\pi$"])

    if col == 0: #if i % 2 == 0: 
        ax_twin.set_ylabel("")
        ax_twin.yaxis.set_ticklabels([])

        # ax_twin.set_yticks([])
        
        set_sci_notation_label(
                target_ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 10, offset_points = (0,30)
            )
    else:
        #target_ax.set_ylabel("")
        ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize = 20)
        ax_twin.set_yticks([-3,-2,-1,0,1,2,3])

    
    #ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize = 25)


    for line in secondary_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties for the twin axis
        ax_twin.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                        linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # # Copy legends
    # if source_ax.get_legend():
    #     target_ax.legend(fontsize='large')

    # if secondary_ax and secondary_ax.get_legend():
    #     ax_twin.legend(fontsize='large')
        
    
    
    #target_ax.set_xlim(min(line.get_xdata())/1e6,max(line.get_xdata())/1e6)
    target_ax.tick_params(axis='x',labelsize=20)
    target_ax.tick_params(axis='y',labelsize=20)
    #target_ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    if row == 0 and col == 0: 
        target_ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))



    #ax_twin.tick_params(axis='x',labelsize=15,length = 10, width = 2)
    ax_twin.tick_params(axis='x',labelsize=20)

    ax_twin.tick_params(axis='y',labelsize=20)
    
    plt.tight_layout()

# Create a figure

fig = plt.figure(figsize=(24, 24))


# Define the GridSpec layout
gs = GridSpec(60, 40, figure=fig)


order = [['NN_fit_comp'],
         ['violin'],
         ['switching_maps']
        ]

subplot_specs = [(0, 30, 0, 20 ), # top left: NN fit comparisons 
                 (0, 29, 20, 40), # g
                 (30, 80, 0, 60), #bottom 
                ]


renderViolin = True
renderSwitchingMaps = False
renderBMWComp = False

for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'violin':
            if renderViolin:
                df = pd.DataFrame()

                # uses the model to get the predictions
                pred_data, scaled_param, params = model.predict(data, is_SHO=False)

                true = dataset.LSQF_hysteresis_params().reshape(-1, 9)

                true_scaled = dataset.loop_param_scaler.transform(true)

                # Builds the dataframe for the violin plot
                true_df = pd.DataFrame(
                    true, columns=["a0", "a1", "a2", "a3", "a4",
                                "b0", "b1", "b2", "b3"]
                )
                predicted_df = pd.DataFrame(
                    scaled_param, columns=["a0", "a1", "a2", "a3", "a4",
                                        "b0", "b1", "b2", "b3"]
                )

                # merges the two dataframes
                df = pd.concat((predicted_df, true_df))

                # adds the labels to the dataframe
                names = [true_scaled, scaled_param]
                names_str = ["NN", "LSQF"]
                labels = ["a0", "a1", "a2", "a3", "a4", "b0", "b1", "b2", "b3"]

                # adds the labels to the dataframe
                for j, name in enumerate(names):
                    for i, label in enumerate(labels):
                        dict_ = {
                            "value": name[:, i],
                            "parameter": np.repeat(label, name.shape[0]),
                            "dataset": np.repeat(names_str[j], name.shape[0]),
                        }

                        df = pd.concat((df, pd.DataFrame(dict_)))
                        
                
                        

    #             # builds the plot
    #             fig, ax = plt.subplots(figsize=(4, 4))

                # Reset index to handle potential duplicated columns or indices
                df = df.reset_index(drop=False)
                
                # plots the data
                sns.violinplot(
                    data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax
                )

                # labels the figure and does some styling
                labelfigs(ax, string_add = 'g', loc ='tl',size=20, style="b", inset_fraction=(0.05,0.12))
                ax.set_ylabel("Scaled Hysteresis Results",fontsize=16)
                ax.set_xlabel("")
                
                ax.tick_params(axis='x',labelsize=14)
                ax.tick_params(axis='y',labelsize=14)

                # Get the legend associated with the plot
                legend = ax.get_legend()
                legend.set_title("")
                plt.setp(legend.get_texts(), fontsize=20) # Set the label size
        
        elif idx[0] == "switching_maps": 
            if renderSwitchingMaps:
                colorbars=True,
                cycle=0,
                fig_width=10.5,  # figure width in inches
                filename=None,

                # # reshape data:
                # if data.shape != 3:
                
                pred_recon, pred_params_scaled, pred_params = model.predict(
                    data,
                    1024,
                    translate_params=False,
                    is_SHO=False
                )

                # calculates the size of the embedding image
                embedding_image_size = 60

                fig, axs = plt.subplots(
                    2,
                    9,
                    figsize=(fig_width, 4),
                    gridspec_kw={"height_ratios": [1, 1]},
                )

                parms_lsqf = dataset.LSQF_hysteresis_params()[:, :, cycle, :].reshape(-1, 9)
                pred_params = pred_params.reshape(embedding_image_size, embedding_image_size, 4, 9)[:, :, cycle, :].reshape(-1, 9)

                clims = []

                colorbar_labels = [
                    'a0', 'a1', 'a2', 'a3', 'a4', 'b0', 'b1', 'b2', 'b3'
                ]

                # Titles for each row
                row_titles = ['Predicted Parameters', 'LSQF Parameters']

                string_add = 'a'

                for i in range(9):
                    clims.append(
                        (
                            np.min(
                                [
                                    pred_params[:, i].min(),
                                    parms_lsqf[:, i].min(),
                                ]
                            ),
                            np.max(
                                [
                                    pred_params[:, i].max(),
                                    parms_lsqf[:, i].max(),
                                ]
                            ),
                        )
                    )

                    axs[0, i].imshow(
                        pred_params[:, i].reshape(
                            embedding_image_size, embedding_image_size),
                        cmap="viridis",
                        vmin=clims[i][0],
                        vmax=clims[i][1],
                    )
                    axs[0,i].set_xticklabels('')
                    axs[0,i].set_yticklabels('')
                    axs[1, i].imshow(
                        parms_lsqf[:, i].reshape(
                            embedding_image_size, embedding_image_size),
                        cmap="viridis",
                        vmin=clims[i][0],
                        vmax=clims[i][1],
                    )
                    axs[1,i].set_xticklabels('')
                    axs[1,i].set_yticklabels('')

                    if colorbars:
                        
                        # Create an axis divider for each subplot
                        divider = make_axes_locatable(axs[1, i])
                        # Append axes to the bottom of the divider with appropriate padding
                        cax = divider.append_axes("bottom", size="5%", pad=0.25) 
                        
                        
                        fmt = ScalarFormatter(useMathText=True)
                        fmt.set_powerlimits((0, 0))
                        cbar = plt.colorbar(axs[1,i].images[0],
                                            cax=cax, format=fmt,orientation = 'horizontal')
                        cbar.set_label(colorbar_labels[i])  # Add a label to the colorbar

                        
                        
                        # cbar = plt.colorbar(
                        #     axs[1, i].images[0], cax=cax, format="%.1e", orientation='horizontal')
                        # # Set the label for each colorbar
                        # cbar.set_label(colorbar_labels[i])

                    labelfigs(axs[0,i],
                            string_add=colorbar_labels[i],
                            loc ='ct',
                            size=8,
                            inset_fraction=(0.2, 0.2)
                            )
                        # Update the char to the next order
                    ascii_value = ord(string_add)+1
                    string_add = chr(ascii_value)

                labelfigs(axs[0,0],
                string_add='a',
                loc ='tl',
                size=8,
                inset_fraction=(0.2, 0.2)
                )
                labelfigs(axs[1,0],
                string_add='b',
                loc ='tl',
                size=8,
                inset_fraction=(0.2, 0.2)
                )

                # Calculate the vertical position for the row titles
                title_y_positions = [0.85, 0.5]  # You may need to adjust these values

                # Set the titles for each row using fig.text
                for i, title in enumerate(row_titles):
                    fig.text(0.5, title_y_positions[i], title, ha='center',
                                va='center', fontsize=10, transform=fig.transFigure)


                # # prints the figure
                # if self.Printer is not None and filename is not None:
                #     print('use printing function')
                #     self.Printer.savefig(
                #         fig, filename, size=6, loc="tl", inset_fraction=(0.2, 0.2)
                #     )
                
                ax.add_child_axes(axs)
                # for row in range(2):
                #     for col in range(9):
                #         inset_ax = ax.inset_axes([-0.06+(col/11.8)+np.floor(col/4)/256,1-(row+1)/6.1-row/192-np.floor(row/2)/96,1/6.1,1/6.1])
                #         inset_ax.imshow(axs[(9*row)+col].get_images()[0].get_array().data)
                        
                # ax.axis("off")


In [ ]:
axs